# Self-Training for Indonesian Tweet Sentiment Classification
## Exploration, Weak Supervision, and Semi-Supervised Pipeline

This notebook walks through the end-to-end research methodology for classifying unlabeled Indonesian tweets using iterative self-training and pseudo-labeling.

### 1. Environment Setup & Library Imports

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sys.path.insert(0, os.path.abspath('..'))
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix


### 2. Dataset Inspection & Quality Audit

Verifying record counts, lack of original sentiment labels, and duplicate statistics.

In [ ]:
with open('../data/processed/audit_summary.json', 'r') as f:
    audit = json.load(f)
for k, v in audit.items():
    print(f'{k:30s}: {v}')


### 3. Text Preprocessing & Slang Normalization

Preserving negations (`tidak`, `bukan`, `belum`), reducing character elongations, and stripping platform noise.

In [ ]:
from src.data_preprocessing import clean_tweet

sample_tweets = [
    'lagu bosan apa yang aku save ni huhuhuhuhuhuhuhuhuhuuuuuuuuuuuuuu',
    'Doa rezeki tak putus Inna haa zaa larizquna maa lahu min na fadesungguhnya',
    'Makasih loh ntar kita bagi hasil aku 99 9 sisanya buat kamu',
    'Aku tak faham betul jenis orang malaysia yang tak reti nak sangkut balik hose tu Aku geli bodoh nak pegang'
]

for t in sample_tweets:
    print('RAW : ', t)
    print('CLEAN:', clean_tweet(t))
    print('-'*60)


### 4. Weak Supervision with InSet Lexicon & Syntactic Negation Windows

Scoring candidate seed labels and creating silver reference annotations without data leakage.

In [ ]:
from src.labeling import InSetSentimentScorer

scorer = InSetSentimentScorer(pos_path='../data/lexicon/inset_positive.tsv', neg_path='../data/lexicon/inset_negative.tsv')

for t in sample_tweets:
    cln = clean_tweet(t)
    res = scorer.score_text(cln)
    print(f"Text: {cln[:40]}... -> Polarity: {res['label']} (Net Score: {res['score']}, Conf: {res['confidence']})")


### 5. Evaluation Splits & Independence

Inspecting the independent test set (500 records) and weak seed training set (3,000 records).

In [ ]:
seed_df = pd.read_csv('../data/processed/train_seed.csv')
test_df = pd.read_csv('../data/processed/test_set.csv')

print('Seed Set Class Distribution:')
print(seed_df['label'].value_counts())
print('\nTest Set Class Distribution (Independent Silver Reference):')
print(test_df['label'].value_counts())


### 6. Baseline vs Self-Trained Model Performance Comparison

In [ ]:
with open('../outputs/metrics/test_metrics_comparison.json', 'r') as f:
    test_metrics = json.load(f)

comp_df = pd.DataFrame({
    'Dummy Majority': [test_metrics['dummy_majority']['accuracy'], test_metrics['dummy_majority']['macro_f1'], test_metrics['dummy_majority']['weighted_f1']],
    'Seed-Only Baseline': [test_metrics['seed_baseline']['accuracy'], test_metrics['seed_baseline']['macro_f1'], test_metrics['seed_baseline']['weighted_f1']],
    'Self-Trained Model': [test_metrics['self_trained_final']['accuracy'], test_metrics['self_trained_final']['macro_f1'], test_metrics['self_trained_final']['weighted_f1']]
}, index=['Accuracy', 'Macro F1', 'Weighted F1'])

comp_df.round(4)


### 7. Performance Visualizations Overview

Embedding generated publication figures.

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/figures/model_comparison.png'))
display(Image(filename='../outputs/figures/confusion_matrix.png'))


### 8. Linguistic Error Breakdown & Qualitative Analysis

In [ ]:
with open('../outputs/metrics/error_analysis.json', 'r') as f:
    err_data = json.load(f)

print(f"Total Errors: {err_data['total_errors']}/{err_data['total_test_samples']} (Error Rate: {err_data['error_rate']*100:.1f}%)")
print("\nError Distribution:")
for cat, cnt in err_data['error_distribution'].items():
    print(f"  {cat:40s}: {cnt}")
